In [ ]:
# --- setup -------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/pxr-repo'
!git clone -q https://github.com/pridem755/patient-or-xray.git $REPO 2>/dev/null || (cd $REPO && git pull -q)
%pip install -q -e $REPO

%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import pandas as pd

from pxr.config import load_config
from pxr.data.cohort import build_cohort, verify_manifest
from pxr.data.contracts import inferential_cell_table, validate_primary_cohort

cfg = load_config(f'{REPO}/config/study_config.yaml')
ROOT = Path(cfg.paths['drive_root'])
META = ROOT / cfg.paths['metadata']
OUT = ROOT / cfg.paths['cohorts']
OUT.mkdir(parents=True, exist_ok=True)

print('config_hash :', cfg.config_hash)
print('labels :', ', '.join(cfg.analysis_labels))
print('sites :', ', '.join(cfg.site_names))

In [ ]:
# --- manifests ----------------------------------------------------------------
MANIFESTS = ROOT / cfg.paths['manifests']

manifests = {}
for site in cfg.site_names:
    path = MANIFESTS / f'{site}_manifest.txt'
    names = [ln.strip() for ln in path.read_text().splitlines() if ln.strip()]
    manifests[site] = names
    print(f'{site:<10} {len(names):>7,} held images   e.g. {names[0]}')

print(f'\ntotal {sum(len(v) for v in manifests.values()):>7,}')

In [ ]:
# --- verify manifests ---------------------------------------------------------
reports = {}
for site in cfg.site_names:
    rep = verify_manifest(site, cfg, META / site, manifests[site])
    reports[site] = rep
    print(f'\n=== {site} ===')
    print(rep.to_string(index=False))

blocking = {s: r[~r.ok & (r.check != 'metadata_covered')] for s, r in reports.items()}
failed = {s: r for s, r in blocking.items() if len(r)}
if failed:
    raise SystemExit(f'STOP — manifest verification failed for: {list(failed)}')
print('\nAll manifests reconcile with metadata.')

In [ ]:
# --- build cohorts ------------------------------------------------------------
results = {}
for site in cfg.site_names:
    print(f'building {site} ...')
    results[site] = build_cohort(site, cfg, META / site, image_manifest=manifests[site])
    print(f'retained {len(results[site].cohort):,} images')

In [ ]:
# --- summarize cohorts --------------------------------------------------------
for site, res in results.items():
    print(f'\n=== {site} ===')
    print(res.flow.to_frame().to_string(index=False))
    if res.warnings:
        print('warnings:')
        for w in res.warnings:
            print('  -', w)

In [ ]:
# exclusion reasons side by side
summary = pd.concat(
    [res.exclusion_summary().assign(site=site) for site, res in results.items()]
).pivot(index='outcome', columns='site', values='n_images').fillna(0).astype(int)
summary.loc['TOTAL'] = summary.sum()
print(summary.to_string())

In [ ]:
# --- validate cohorts ---------------------------------------------------------
for site, res in results.items():
    obs = [cfg.harmonisation(site).get(x, x) for x in cfg.observation_schema(site)]
    report = validate_primary_cohort(
        res.cohort,
        cfg.analysis_labels,
        observation_labels=obs,
        artifact=f'cohort_{site}',
        expected_site=site,
        expected_config_hash=cfg.config_hash,
        min_positives=cfg.min_positives_warning,
        strata=cfg.strata,
        strict=False,
    )
    print(report.summary(max_warnings=5))
    print()
    assert report.ok, f'{site} failed its contract'

In [ ]:
# --- cohort summaries ---------------------------------------------------------
rows = []
for site, res in results.items():
    c = res.cohort
    rows.append({
        'site': site,
        'patients': f'{len(c):,}',
        'AP': f'{(c.view == "AP").mean():.1%}',
        'female': f'{(c.sex == "Female").mean():.1%}',
        'age median': f'{c.age.median():.0f}',
        'No Finding': f'{(c["No Finding"] == 1).mean():.1%}',
    })
print(pd.DataFrame(rows).to_string(index=False))

print('\nP(AP) by age band  <-- the coupling this study is about:')
print(pd.concat([
    res.cohort.assign(site=site).groupby(['site', 'age_bin'], observed=True)
       .view.apply(lambda v: (v == 'AP').mean())
    for site, res in results.items()
]).unstack().round(3).to_string())

print('\nP(AP) by sex:')
print(pd.concat([
    res.cohort.assign(site=site).groupby(['site', 'sex'], observed=True)
       .view.apply(lambda v: (v == 'AP').mean())
    for site, res in results.items()
]).unstack().round(3).to_string())

In [ ]:
# --- cohort summaries: No Finding counts ---------------------------------------
for site, res in results.items():
    c = res.cohort
    nf = c['No Finding']
    print(f'{site:<10} No Finding=1 {int((nf == 1).sum()):>7,}   '
          f'No Finding=0 {int((nf == 0).sum()):>7,}')

In [ ]:
# positives per label x view — the raw material for the power gate in notebook 03
for site, res in results.items():
    c = res.cohort
    counts = pd.DataFrame({
        v: [int((c.loc[c.view == v, lab] == 1).sum()) for lab in cfg.analysis_labels]
        for v in ('AP', 'PA')
    }, index=cfg.analysis_labels)
    print(f'\n=== {site}: positive patients ===')
    print(counts.to_string())

In [ ]:
# --- save cohorts -------------------------------------------------------------
for site, res in results.items():
    cohort_path = OUT / cfg.artifact_name('cohort', site=site)
    res.cohort.to_parquet(cohort_path, index=False)
    res.flow.to_frame().to_csv(OUT / cfg.artifact_name('flow', site=site, ext='csv'), index=False)
    res.audit.to_parquet(OUT / cfg.artifact_name('audit', site=site), index=False)
    print(f'{site:<10} -> {cohort_path.name}')

pd.concat([res.flow.to_frame() for res in results.values()]).to_csv(
    OUT / f'cohort_flow_all_{cfg.config_hash}.csv', index=False
)

In [ ]:
# --- cohort summaries: final table ---------------------------------------------
print(f'config_hash: {cfg.config_hash}')
print(f'{"site":<12}{"held":>10}{"retained":>10}{"patients":>10}{"AP%":>8}')
for site, res in results.items():
    c = res.cohort
    print(f'{site:<12}{len(manifests[site]):>10,}{len(c):>10,}'
          f'{c.patient_id.nunique():>10,}{(c.view == "AP").mean():>8.1%}')

total = sum(len(r.cohort) for r in results.values())
print(f'\nanalysis cohort: {total:,} patients '
      f'({sum(len(m) for m in manifests.values()):,} held)')

cells_tab = pd.concat([
    inferential_cell_table(res.cohort, cfg.analysis_labels, strata=cfg.strata).assign(site=site)
    for site, res in results.items()
])
thin = cells_tab[cells_tab.n_positive < cfg.min_positives_warning]
print(f'inferential cells: {len(cells_tab):,} total, '
      f'{len(thin):,} below {cfg.min_positives_warning} positives '
      f'(the power gate in notebook 03 decides how these are handled)')